# FER2013 Training Experiments

This notebook is designed for Colab or Kaggle GPU training. It uses the dataset analysis from `01_data_exploration.ipynb` to justify the experiment protocol: a CNN trained from scratch for native 48x48 grayscale inputs, augmentation for low-resolution face variability, ResNet18 transfer learning for pretrained visual features, and imbalance-aware training for uneven emotion counts.


## 1. Setup

For Colab, clone your GitHub repository or upload it to Drive. For Kaggle, add the FER2013 image-folder dataset and set `DATA_DIR` to the Kaggle input path.


In [ ]:
# Optional in Colab/Kaggle if dependencies are missing:
# !pip install -q torch torchvision pandas scikit-learn matplotlib seaborn tqdm

In [ ]:
# Comment this section if you are running your notebook in kaggle
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'src'))

# Local path after extracting Kaggle data:
DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'fer2013_images'

# Kaggle example, uncomment and edit if needed:
# DATA_DIR = Path('/kaggle/input/fer2013')

RESULTS_DIR = PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)
DATA_DIR


### Kaggle Path Setup Option

Use this block only when running the notebook in Kaggle. Keep the local setup cell above for local runs. In Kaggle, comment out the local `PROJECT_ROOT`, `sys.path`, and `DATA_DIR` lines above, then uncomment and edit the Kaggle paths below. The Kaggle code path should point to the cloned repository, while `DATA_DIR` should point to the Kaggle FER2013 dataset folder.


In [ ]:
# Kaggle option. Uncomment this block only when running in Kaggle.

%cd /kaggle/working
import os
from pathlib import Path
# !git clone https://github.com/zsykk/DL-final-project.git
REPO_URL = "https://github.com/zsykk/DL-final-project.git"
REPO_DIR = "DL-final-project"

if Path(REPO_DIR).exists():
    %cd /kaggle/working/DL-final-project
    !git pull
else:
    !git clone {REPO_URL}
    %cd /kaggle/working/DL-final-project


from pathlib import Path
import sys

PROJECT_ROOT = Path('/kaggle/working/DL-final-project')  # cloned repo folder
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

DATA_DIR = Path('/kaggle/input/datasets/msambare/fer2013')  # edit if your Kaggle dataset folder name differs
RESULTS_DIR = PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

DATA_DIR, (DATA_DIR / "train").exists(), (DATA_DIR / "test").exists()
PROJECT_ROOT, PROJECT_ROOT.exists(), (PROJECT_ROOT / 'src' / 'fer_project').exists(), DATA_DIR, DATA_DIR.exists()


In [ ]:
import importlib
import random
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 42
set_seed(SEED)

from fer_project.data import TransformConfig, build_imagefolder_dataloaders, class_weights, dataset_labels
import fer_project.metrics as metrics

metrics = importlib.reload(metrics)
collect_predictions = metrics.collect_predictions
plot_confusion_matrix = metrics.plot_confusion_matrix
save_classification_report = metrics.save_classification_report
top_confusions = metrics.top_confusions

from fer_project.models import build_model
import fer_project.training as training
training = importlib.reload(training)
fit = training.fit
fit_adamw_two_stage = training.fit_adamw_two_stage

import fer_project.data as data
data = importlib.reload(data)

import fer_project.metrics as metrics
metrics = importlib.reload(metrics)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE


## 2. Experiment Strategy

Each stage defines its own configuration directly before running. This keeps the notebook iterative: run one experiment, inspect validation behavior, macro/weighted F1, per-class metrics, and confusions, then adjust the next experiment instead of relying on one large configuration block.


### Why ResNet18 and MobileNetV2?

The dataset analysis shows that FER2013 images are small, grayscale, imbalanced, and visually ambiguous. A custom CNN is useful because it matches the native 48x48 grayscale input and gives a fair from-scratch baseline. This baseline still uses standard deep-learning improvements: BatchNorm stabilizes activations and helps optimization, Dropout/Dropout2d reduces overfitting, ReLU adds nonlinearity, MaxPool gradually reduces spatial size, AdamW adds decoupled weight decay, and the separate augmentation/class-weight/sampler/focal-loss experiments test common regularization and imbalance strategies. Therefore, the baseline is a practical optimized CNN baseline, not a deliberately simple straw-man model.

ResNet18 is the main pretrained model because it is a standard, moderate-size ImageNet backbone with residual skip connections. The residual design makes it easier to train deeper convolutional models than plain CNNs, while the ImageNet pretraining gives the network reusable low- and mid-level visual features such as edges, corners, textures, contrast patterns, and local shapes. These features are not specific to ImageNet classes; they are also useful for facial-expression images, where the model must detect visual patterns around the eyes, mouth, eyebrows, and face outline. ResNet18 is therefore a reasonable transfer-learning choice for FER2013: it is expressive enough to improve over a small custom CNN, but not as computationally heavy or over-parameterized as larger ResNets such as ResNet50 or ResNet101 for this project scale.

A model such as ResNet15 is not selected because it is not a standard pretrained torchvision model. Using it would either require training from scratch or using a custom checkpoint, which would weaken the comparison and make the experiment less reproducible. ResNet18 is preferred because its pretrained weights are readily available, well documented, and commonly used as a baseline in transfer-learning experiments.

MobileNetV2 is included as a second pretrained strategy, not as a replacement for ResNet18. Its purpose is different: it is a lightweight architecture designed for efficiency, so it lets the report compare a standard transfer backbone against a smaller transfer backbone. If MobileNetV2 performs close to ResNet18, that is useful evidence for computational efficiency; if it performs worse, that supports using the stronger ResNet18 model.

For both pretrained models, FER2013 images are converted from 48x48 grayscale to 224x224 RGB tensors with ImageNet normalization. Frozen experiments test whether fixed pretrained features are useful; fine-tuned experiments test whether adapting the whole backbone improves performance on FER2013.


## 3. Shared Experiment Runner

The helper below trains one configuration, saves the best validation-loss checkpoint, evaluates on the test set, and plots loss plus confusion matrix in one compact row.


### Loss Curve Helper

Use this after each experiment to inspect convergence and possible overfitting from train/validation loss.


In [ ]:
def plot_loss_curve(history, title):
    ax = history.plot(
        x='epoch',
        y=['train_loss', 'val_loss'],
        marker='o',
        figsize=(5, 3),
        title=title,
    )
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    plt.tight_layout()
    plt.show()
    return ax


def plot_training_and_confusion(history, y_true, y_pred, title, confusion_output_path=None):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
    history.plot(
        x='epoch',
        y=['train_loss', 'val_loss'],
        marker='o',
        ax=axes[0],
    )
    axes[0].set_title('Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')

    plot_confusion_matrix(
        y_true,
        y_pred,
        output_path=confusion_output_path,
        figsize=(4.8, 3.8),
        ax=axes[1],
    )
    axes[1].set_title('Confusion Matrix')
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()
    return fig, axes


In [ ]:
def run_experiment(config, batch_size=128, num_workers=0, subset_fraction=1.0):
    name = config['name']
    loaders, datasets = build_imagefolder_dataloaders(
        DATA_DIR,
        train_config=config['train_config'],
        eval_config=config['eval_config'],
        batch_size=batch_size,
        num_workers=num_workers,
        weighted_sampler=config.get('weighted_sampler', False),
        val_fraction=0.1,
        subset_fraction=subset_fraction,
        seed=42,
    )
    if config['model_kind'] == 'baseline_cnn':
        model = build_model('baseline_cnn')
    else:
        model = build_model(
            'transfer',
            transfer_model=config.get('transfer_model', 'resnet18'),
            freeze_backbone=config.get('freeze_backbone', True),
        )

    weights = class_weights(dataset_labels(datasets['train'])) if config.get('use_class_weights') else None
    checkpoint_path = RESULTS_DIR / 'checkpoints' / f'{name}.pt'
    history = fit(
        model,
        loaders,
        device=DEVICE,
        epochs=config['epochs'],
        lr=config['lr'],
        class_weight=weights,
        loss_name=config.get('loss_name', 'cross_entropy'),
        focal_gamma=config.get('focal_gamma', 2.0),
        checkpoint_path=checkpoint_path,
    )

    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
    y_true, y_pred = collect_predictions(model, loaders['test'], DEVICE)

    report_path = RESULTS_DIR / 'metrics' / f'{name}_classification_report.csv'
    report = save_classification_report(y_true, y_pred, report_path)
    confusions = top_confusions(y_true, y_pred, top_n=10)
    confusions.to_csv(RESULTS_DIR / 'metrics' / f'{name}_top_confusions.csv', index=False)
    history_frame = pd.DataFrame(history)
    history_frame.to_csv(RESULTS_DIR / 'metrics' / f'{name}_history.csv', index=False)
    plot_training_and_confusion(
        history_frame,
        y_true,
        y_pred,
        f'{name}: loss and confusion matrix',
        RESULTS_DIR / 'figures' / f'{name}_confusion_matrix.png',
    )
    return history_frame, report, confusions


## 4. Run Experiments Safely

Run one stage at a time. After each stage, inspect the validation curve, macro F1, per-class metrics, and top confusions before deciding whether the next tuning step is justified. This supports the report narrative: dataset observation, tuning choice, result, and conclusion. Each run saves outputs under `results/`, so you can merge completed results later without retraining everything.


In [ ]:
EXPERIMENT_REGISTRY = {}


def register_experiment(config):
    EXPERIMENT_REGISTRY[config['name']] = config
    return config


def run_config_stage(config, batch_size=128, num_workers=0, subset_fraction=1.0):
    config = register_experiment(config)
    history, report, confusions = run_experiment(
        config,
        batch_size=batch_size,
        num_workers=num_workers,
        subset_fraction=subset_fraction,
    )
    display(report.loc[['macro avg', 'weighted avg']])
    display(report.loc[['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise', 'neutral'], ['precision', 'recall', 'f1-score', 'support']])
    display(confusions)
    return history, report, confusions


### Stage 1: Augmented Baseline CNN

Start with the self-defined CNN using augmentation as part of the default baseline. The data exploration showed that FER2013 images are small, low-resolution, and visually variable, so flips, small rotations, and translations are treated as a necessary regularization choice rather than an optional trick.


In [ ]:
baseline_cnn_aug_config = {
    'name': 'baseline_cnn_aug',
    'group': 'required_baseline',
    'model_kind': 'baseline_cnn',
    'train_config': TransformConfig(image_size=48, channels=1, augment=True),
    'eval_config': TransformConfig(image_size=48, channels=1, augment=False),
    'lr': 1e-3,
    'epochs': 20,
    'use_class_weights': False,
    'weighted_sampler': False,
    'loss_name': 'cross_entropy',
}

history_aug, report_aug, confusions_aug = run_config_stage(baseline_cnn_aug_config)


### Stage 2: Add Class Weights

Run this if the augmented baseline shows weak minority-class performance. The reason for this tuning step is class imbalance, especially the small `disgust` class.


In [ ]:
baseline_cnn_aug_class_weights_config = {
    'name': 'baseline_cnn_aug_class_weights',
    'group': 'required_baseline',
    'model_kind': 'baseline_cnn',
    'train_config': TransformConfig(image_size=48, channels=1, augment=True),
    'eval_config': TransformConfig(image_size=48, channels=1, augment=False),
    'lr': 1e-3,
    'epochs': 20,
    'use_class_weights': True,
    'weighted_sampler': False,
    'loss_name': 'cross_entropy',
}

history_weight, report_weight, confusions_weight = run_config_stage(baseline_cnn_aug_class_weights_config)


### Stage 3: Optional Imbalance Alternatives

Use these only if class weights are not enough. Run one option at a time, then compare the saved results. Weighted sampling changes which examples appear in batches; focal loss changes the loss so harder examples receive more emphasis.


In [ ]:
baseline_cnn_aug_weighted_sampler_config = {
    'name': 'baseline_cnn_aug_weighted_sampler',
    'group': 'optional_imbalance',
    'model_kind': 'baseline_cnn',
    'train_config': TransformConfig(image_size=48, channels=1, augment=True),
    'eval_config': TransformConfig(image_size=48, channels=1, augment=False),
    'lr': 1e-3,
    'epochs': 20,
    'use_class_weights': False,
    'weighted_sampler': True,
    'loss_name': 'cross_entropy',
}

history_sampler, report_sampler, confusions_sampler = run_config_stage(baseline_cnn_aug_weighted_sampler_config)


In [ ]:
baseline_cnn_aug_focal_loss_config = {
    'name': 'baseline_cnn_aug_focal_loss',
    'group': 'optional_imbalance',
    'model_kind': 'baseline_cnn',
    'train_config': TransformConfig(image_size=48, channels=1, augment=True),
    'eval_config': TransformConfig(image_size=48, channels=1, augment=False),
    'lr': 1e-3,
    'epochs': 20,
    'use_class_weights': False,
    'weighted_sampler': False,
    'loss_name': 'focal',
    'focal_gamma': 2.0,
}

history_focal, report_focal, confusions_focal = run_config_stage(baseline_cnn_aug_focal_loss_config)


### Stage 4: Advanced Baseline Loss and Sampling Experiments

Run these after the required baseline variants. They test longer training and combinations of class weighting, weighted sampling, and focal loss.


In [ ]:
baseline_cnn_aug_class_weights_focal_config = {
    'name': 'baseline_cnn_aug_class_weights_focal_loss',
    'group': 'advanced_baseline',
    'model_kind': 'baseline_cnn',
    'train_config': TransformConfig(image_size=48, channels=1, augment=True),
    'eval_config': TransformConfig(image_size=48, channels=1, augment=False),
    'lr': 1e-3,
    'epochs': 30,
    'use_class_weights': True,
    'weighted_sampler': False,
    'loss_name': 'focal',
    'focal_gamma': 2.0,
}

history_weight_focal, report_weight_focal, confusions_weight_focal = run_config_stage(
    baseline_cnn_aug_class_weights_focal_config
)

# Alternative to try after inspecting the result above:
# baseline_cnn_aug_weighted_sampler_focal_config = {
#     'name': 'baseline_cnn_aug_weighted_sampler_focal_loss',
#     'group': 'advanced_baseline',
#     'model_kind': 'baseline_cnn',
#     'train_config': TransformConfig(image_size=48, channels=1, augment=True),
#     'eval_config': TransformConfig(image_size=48, channels=1, augment=False),
#     'lr': 1e-3,
#     'epochs': 30,
#     'use_class_weights': False,
#     'weighted_sampler': True,
#     'loss_name': 'focal',
#     'focal_gamma': 2.0,
# }
# history_sampler_focal, report_sampler_focal, confusions_sampler_focal = run_config_stage(
#     baseline_cnn_aug_weighted_sampler_focal_config
# )


In [ ]:
from fer_project.data import build_baseline_crop_dataloaders
from fer_project.metrics import collect_predictions_with_crops
from fer_project.training import fit_sgd_schedule, fit_adamw_two_stage


def finalize_baseline_run(model, loaders, history, name, checkpoint_path):
    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
    y_true, y_pred = collect_predictions_with_crops(model, loaders['test'], DEVICE)

    report = save_classification_report(y_true, y_pred, RESULTS_DIR / 'metrics' / f'{name}_classification_report.csv')
    confusions = top_confusions(y_true, y_pred, top_n=10)
    confusions.to_csv(RESULTS_DIR / 'metrics' / f'{name}_top_confusions.csv', index=False)
    history_frame = pd.DataFrame(history)
    history_frame.to_csv(RESULTS_DIR / 'metrics' / f'{name}_history.csv', index=False)
    plot_training_and_confusion(
        history_frame,
        y_true,
        y_pred,
        f'{name}: loss and confusion matrix',
        RESULTS_DIR / 'figures' / f'{name}_confusion_matrix.png',
    )
    display(report.loc[['macro avg', 'weighted avg']])
    display(report.loc[['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise', 'neutral'], ['precision', 'recall', 'f1-score', 'support']])
    display(confusions)
    return history_frame, report, confusions


def run_baseline_sgd_schedule(
    name='baseline_cnn_aug_sgd_clip_lr_decay',
    batch_size=128,
    num_workers=2,
    subset_fraction=1.0,
    epochs=60,
    lr=0.01,
    use_class_weights=False,
    loss_name='cross_entropy',
):
    loaders, datasets = build_imagefolder_dataloaders(
        DATA_DIR,
        train_config=TransformConfig(image_size=48, channels=1, augment=True),
        eval_config=TransformConfig(image_size=48, channels=1, augment=False),
        batch_size=batch_size,
        num_workers=num_workers,
        weighted_sampler=False,
        val_fraction=0.1,
        subset_fraction=subset_fraction,
        seed=SEED,
    )
    model = build_model('baseline_cnn')
    weights = class_weights(dataset_labels(datasets['train'])) if use_class_weights else None
    checkpoint_path = RESULTS_DIR / 'checkpoints' / f'{name}.pt'

    history, _ = fit_sgd_schedule(
        model,
        loaders,
        device=DEVICE,
        epochs=epochs,
        lr=lr,
        class_weight=weights,
        loss_name=loss_name,
        checkpoint_path=checkpoint_path,
        stage_name='single_stage',
    )
    return finalize_baseline_run(model, loaders, history, name, checkpoint_path)


def run_baseline_crop_two_stage_sgd(
    name='baseline_cnn_crop_tencrop_two_stage_sgd',
    batch_size=128,
    num_workers=2,
    subset_fraction=1.0,
    stage1_epochs=40,
    stage2_epochs=15,
    stage1_lr=0.01,
    stage2_lr=0.001,
    crop_size=44,
    ten_crop_eval=True,
    use_class_weights=False,
    weighted_sampler=False,
    loss_name='cross_entropy',
):
    loaders, datasets = build_baseline_crop_dataloaders(
        DATA_DIR,
        batch_size=batch_size,
        num_workers=num_workers,
        subset_fraction=subset_fraction,
        seed=SEED,
        image_size=48,
        crop_size=crop_size,
        ten_crop_eval=ten_crop_eval,
        weighted_sampler=weighted_sampler,
    )
    model = build_model('baseline_cnn')
    weights = class_weights(dataset_labels(datasets['train'])) if use_class_weights else None
    checkpoint_path = RESULTS_DIR / 'checkpoints' / f'{name}.pt'

    history_stage1, best_val_loss = fit_sgd_schedule(
        model,
        loaders,
        device=DEVICE,
        epochs=stage1_epochs,
        lr=stage1_lr,
        class_weight=weights,
        loss_name=loss_name,
        checkpoint_path=checkpoint_path,
        initial_best_val_loss=float('inf'),
        epoch_offset=0,
        stage_name='stage1_adapt',
    )

    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))

    history_stage2, best_val_loss = fit_sgd_schedule(
        model,
        loaders,
        device=DEVICE,
        epochs=stage2_epochs,
        lr=stage2_lr,
        class_weight=weights,
        loss_name=loss_name,
        checkpoint_path=checkpoint_path,
        initial_best_val_loss=best_val_loss,
        epoch_offset=stage1_epochs,
        stage_name='stage2_lower_lr',
    )

    history = history_stage1 + history_stage2
    return finalize_baseline_run(model, loaders, history, name, checkpoint_path)



def run_transfer_two_stage(
    name,
    transfer_model='resnet18',
    batch_size=64,
    num_workers=2,
    subset_fraction=1.0,
    stage1_epochs=10,
    stage2_epochs=10,
    stage1_lr=1e-4,
    stage2_lr=1e-5,
    use_class_weights=False,
    loss_name='cross_entropy',
):
    train_config = TransformConfig(image_size=224, channels=3, augment=True, imagenet_norm=True)
    eval_config = TransformConfig(image_size=224, channels=3, augment=False, imagenet_norm=True)
    loaders, datasets = build_imagefolder_dataloaders(
        DATA_DIR,
        train_config=train_config,
        eval_config=eval_config,
        batch_size=batch_size,
        num_workers=num_workers,
        weighted_sampler=False,
        val_fraction=0.1,
        subset_fraction=subset_fraction,
        seed=SEED,
    )
    model = build_model(
        'transfer',
        transfer_model=transfer_model,
        freeze_backbone=False,
    )
    weights = class_weights(dataset_labels(datasets['train'])) if use_class_weights else None
    checkpoint_path = RESULTS_DIR / 'checkpoints' / f'{name}.pt'

    history = fit_adamw_two_stage(
        model,
        loaders,
        device=DEVICE,
        stage1_epochs=stage1_epochs,
        stage2_epochs=stage2_epochs,
        stage1_lr=stage1_lr,
        stage2_lr=stage2_lr,
        class_weight=weights,
        loss_name=loss_name,
        checkpoint_path=checkpoint_path,
    )

    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
    y_true, y_pred = collect_predictions(model, loaders['test'], DEVICE)

    report = save_classification_report(y_true, y_pred, RESULTS_DIR / 'metrics' / f'{name}_classification_report.csv')
    confusions = top_confusions(y_true, y_pred, top_n=10)
    confusions.to_csv(RESULTS_DIR / 'metrics' / f'{name}_top_confusions.csv', index=False)

    history_frame = pd.DataFrame(history)
    history_frame.to_csv(RESULTS_DIR / 'metrics' / f'{name}_history.csv', index=False)
    plot_training_and_confusion(
        history_frame,
        y_true,
        y_pred,
        f'{name}: loss and confusion matrix',
        RESULTS_DIR / 'figures' / f'{name}_confusion_matrix.png',
    )
    display(report.loc[['macro avg', 'weighted avg']])
    display(report.loc[['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise', 'neutral'], ['precision', 'recall', 'f1-score', 'support']])
    display(confusions)
    return history_frame, report, confusions


### Stage 5: Baseline With SGD Optimizer Schedule

This experiment tests the teacher-baseline-style optimizer recipe on our self-defined CNN: SGD, momentum, weight decay, gradient clipping, and learning-rate decay.


In [ ]:
history_sgd, report_sgd, confusions_sgd = run_baseline_sgd_schedule(
    name='baseline_cnn_aug_sgd_clip_lr_decay',
    batch_size=128,
    num_workers=2,
    subset_fraction=1.0,
    epochs=60,
    lr=0.01,
    use_class_weights=False,
    loss_name='cross_entropy',
)


### Stage 6: Baseline With Crop and TenCrop Evaluation

This experiment adds random crop augmentation during training and TenCrop-style prediction averaging during validation/test. It also uses two-stage training: save the best stage-1 checkpoint, reload it, then continue with a lower learning rate.


In [ ]:
history_crop, report_crop, confusions_crop = run_baseline_crop_two_stage_sgd(
    name='baseline_cnn_crop_tencrop_two_stage_sgd',
    batch_size=128,
    num_workers=2,
    subset_fraction=1.0,
    stage1_epochs=40,
    stage2_epochs=15,
    stage1_lr=0.01,
    stage2_lr=0.001,
    crop_size=44,
    ten_crop_eval=True,
    use_class_weights=False,
    weighted_sampler=False,
    loss_name='cross_entropy',
)


### Stage 7: Transfer Learning Comparison

Run transfer learning after the self-defined CNN tuning path. Start with the frozen ResNet18 classifier to test fixed pretrained features, then fine-tune ResNet18 to test whether adapting the backbone improves FER2013 performance.


In [ ]:
resnet18_frozen_config = {
    'name': 'resnet18_frozen',
    'group': 'required_transfer',
    'model_kind': 'transfer',
    'transfer_model': 'resnet18',
    'freeze_backbone': True,
    'train_config': TransformConfig(image_size=224, channels=3, augment=True, imagenet_norm=True),
    'eval_config': TransformConfig(image_size=224, channels=3, augment=False, imagenet_norm=True),
    'lr': 1e-3,
    'epochs': 20,
    'use_class_weights': False,
    'weighted_sampler': False,
    'loss_name': 'cross_entropy',
}

history_resnet_frozen, report_resnet_frozen, confusions_resnet_frozen = run_config_stage(
    resnet18_frozen_config,
    batch_size=128,
    num_workers=2,
)


In [ ]:
resnet18_finetune_config = {
    'name': 'resnet18_finetune',
    'group': 'required_transfer',
    'model_kind': 'transfer',
    'transfer_model': 'resnet18',
    'freeze_backbone': False,
    'train_config': TransformConfig(image_size=224, channels=3, augment=True, imagenet_norm=True),
    'eval_config': TransformConfig(image_size=224, channels=3, augment=False, imagenet_norm=True),
    'lr': 1e-4,
    'epochs': 20,
    'use_class_weights': False,
    'weighted_sampler': False,
    'loss_name': 'cross_entropy',
}

history_resnet_finetune, report_resnet_finetune, confusions_resnet_finetune = run_config_stage(
    resnet18_finetune_config,
    batch_size=64,
    num_workers=2,
)


### Stage 8: Two-Stage Transfer Fine-Tuning

These experiments formalize the save-best-then-continue strategy for transfer models. Stage 1 adapts the full pretrained backbone, then stage 2 reloads the best checkpoint and continues with a lower learning rate.


In [ ]:
history_resnet_two_stage, report_resnet_two_stage, confusions_resnet_two_stage = run_transfer_two_stage(
    name='resnet18_finetune_two_stage',
    transfer_model='resnet18',
    batch_size=64,
    num_workers=2,
    subset_fraction=1.0,
    stage1_epochs=10,
    stage2_epochs=10,
    stage1_lr=1e-4,
    stage2_lr=1e-5,
)


In [ ]:
history_mobilenet_two_stage, report_mobilenet_two_stage, confusions_mobilenet_two_stage = run_transfer_two_stage(
    name='mobilenet_v2_finetune_two_stage',
    transfer_model='mobilenet_v2',
    batch_size=64,
    num_workers=2,
    subset_fraction=1.0,
    stage1_epochs=10,
    stage2_epochs=10,
    stage1_lr=1e-4,
    stage2_lr=1e-5,
)


### Merge Finished Results

Use this only after you have finished several separate runs. It reads saved metric files and displays one comparison table without retraining. The purpose is convenience, not replacing the visible F1/loss outputs above.


In [ ]:
summary_rows = {}
metrics_dir = RESULTS_DIR / 'metrics'

summary_configs = list(EXPERIMENT_REGISTRY.values()) if 'EXPERIMENT_REGISTRY' in globals() else []
summary_configs = summary_configs + [
    {'name': 'baseline_cnn_aug_sgd_clip_lr_decay', 'group': 'advanced_baseline'},
    {'name': 'baseline_cnn_crop_tencrop_two_stage_sgd', 'group': 'advanced_baseline'},
    {'name': 'resnet18_finetune_two_stage', 'group': 'transfer_two_stage'},
    {'name': 'mobilenet_v2_finetune_two_stage', 'group': 'transfer_two_stage'},
]
seen = set()
summary_configs = [config for config in summary_configs if not (config['name'] in seen or seen.add(config['name']))]

for config in summary_configs:
    name = config['name']
    report_path = metrics_dir / f'{name}_classification_report.csv'
    history_path = metrics_dir / f'{name}_history.csv'
    confusions_path = metrics_dir / f'{name}_top_confusions.csv'
    if not report_path.exists() or not history_path.exists():
        continue

    report = pd.read_csv(report_path, index_col=0)
    history = pd.read_csv(history_path)
    confusions = pd.read_csv(confusions_path) if confusions_path.exists() else pd.DataFrame()
    summary_rows[name] = {
        'group': config.get('group', 'unknown'),
        'final_val_loss': float(history['val_loss'].iloc[-1]),
        'best_val_loss': float(history['val_loss'].min()),
        'final_val_accuracy': float(history['val_accuracy'].iloc[-1]),
        'best_val_accuracy': float(history['val_accuracy'].max()),
        'test_macro_f1': float(report.loc['macro avg', 'f1-score']),
        'test_weighted_f1': float(report.loc['weighted avg', 'f1-score']),
        'top_confusion': 'none' if confusions.empty else f"{confusions.iloc[0]['true_emotion']} -> {confusions.iloc[0]['predicted_emotion']}",
    }

summary = pd.DataFrame(summary_rows).T.sort_values('test_macro_f1', ascending=False)
summary.to_csv(metrics_dir / 'experiment_summary.csv')
summary


## 6. Report Checklist

- Start the report from dataset evidence: 48x48 grayscale images, class imbalance, visual ambiguity, and train/test split sizes.
- Treat the required baseline and transfer stages as the minimum protocol for the instructor feedback.
- Present the self-defined CNN as a staged tuning path: augmented baseline CNN, class weighting, optional imbalance alternatives, optimizer schedule, and crop/TenCrop evaluation.
- Explain why ResNet18 is the main pretrained backbone: standard, moderate-size, ImageNet-pretrained, and practical for frozen-vs-fine-tuned transfer learning.
- Explain why MobileNetV2 is added: a lightweight pretrained backbone for efficiency-oriented comparison against ResNet18.
- If time allows, run the optional imbalance, advanced baseline, and two-stage transfer cells to strengthen the comparison.
- Select the best model by macro F1 because FER2013 is imbalanced.
- Use weighted F1 and per-class precision/recall/F1 as supporting metrics.
- Do not use accuracy for model selection or report conclusions.
- Use confusion matrices and each `_top_confusions.csv` file to discuss confused emotions and failure cases.
- Explain the transfer learning input adaptation: 48x48 grayscale images are converted to 3-channel 224x224 tensors with ImageNet normalization.
- Discuss which modeling choice improved minority or ambiguous classes, even if total accuracy did not improve.
